# Practice 1 · MNIST digit classifier

**Interviewer:** "This notebook trains a small CNN on MNIST. The training loss looks odd and the accuracy numbers don't make sense. Please walk me through it, tell me what you're checking, and fix it."

There are **9 problems**: 3 crash the code, the rest are silent (the code runs, but learns badly or reports wrong numbers).

**How to work**
1. Read the whole notebook first and say out loud what each part does.
2. Run the cells and fix crash errors one at a time. Read each error message: it tells you where to look.
3. Once it runs, go through it again for the silent bugs.

*Runtime → Change runtime type → T4 GPU.*

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

BATCH_SIZE = 128
EPOCHS = 3
LR = 0.01

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

### Data

In [ ]:
transform = transforms.Compose([
    transforms.Normalize((0.1307,), (0.3081,)),
    transforms.ToTensor(),
])

train_ds = datasets.MNIST("/content/data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST("/content/data", train=True, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)
print("train:", len(train_ds), "test:", len(test_ds))

### Model

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.pool = nn.MaxPool2d(2)
        self.dropout = nn.Dropout(0.25)
        self.fc1 = nn.Linear(64 * 14 * 14, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = self.dropout(x)
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        return F.relu(self.fc2(x))


model = Net().to(device)
criterion = nn.NLLLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9)

### Training and evaluation

In [ ]:
history = []
model.train()
for epoch in range(EPOCHS):
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.zero_grad()
        optimizer.step()
        running += loss
    history.append(running / len(train_loader))

    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(dim=1)
            correct += (preds == labels).sum().item()
    acc = correct / len(test_loader)
    print(f"epoch {epoch + 1}: train loss {history[-1]:.3f} | test acc {acc:.3f}")

### Plot

In [ ]:
plt.plot(history, marker="o")
plt.xlabel("epoch")
plt.ylabel("train loss")
plt.show()

### Your notes
> Bug: cell … → why it's wrong → my fix → crash or silent?